## Task 0: Setting up Ollama

In [30]:
import requests

response = requests.post(
    "http://localhost:11434/api/generate",
    json={
        "model": "gemma3:270m",
        "prompt": "Explain artificial intelligence in one sentence.",
        "stream": False   
    }
)

data = response.json()
print(data["response"])

Artificial intelligence is the ability of machines to mimic human intelligence processes.



In [32]:
response = requests.post(
    "http://localhost:11434/api/generate",
    json={
        "model": "gemma3:4b",
        "prompt": "Explain neural networks.",
        "stream": False   
    }
)

print(response.json()["response"])

Okay, let's break down neural networks in a way that's hopefully easy to understand. They're a fascinating and powerful technology behind many modern AI systems, like image recognition, language translation, and self-driving cars.

**1. The Basic Idea: Mimicking the Brain**

At their core, neural networks are inspired by the structure and function of the human brain.  Our brains are made up of billions of interconnected neurons that transmit signals to each other.  Neural networks try to mimic this process to learn and solve problems.

**2. The Building Blocks: Neurons (Nodes)**

* **Artificial Neurons:** These are the fundamental units of a neural network. They're simplified mathematical models of biological neurons.
* **Inputs:** Each neuron receives inputs from other neurons or directly from the data.
* **Weights:** Each input has an associated *weight*. Think of weights like how important a particular piece of information is to the neuron.  A higher weight means that input has a bi

## Task 1: Text Classification with Ollam

In [5]:
import pandas as pd

df = pd.read_csv("emails.csv", sep=";")
df.head()


,headline
0,URGENT: Your account will be suspended within ...
1,Congratulations! You have won a 1000€ gift car...
2,Hot singles in your area are waiting to meet y...
3,Re: Inheritance transfer of 4.5M USD pending y...
4,Meeting agenda for Thursday's project review


In [7]:
print(df.columns)

Index(['headline'], dtype='object')


In [9]:
df.head()


,headline
0,URGENT: Your account will be suspended within ...
1,Congratulations! You have won a 1000€ gift car...
2,Hot singles in your area are waiting to meet y...
3,Re: Inheritance transfer of 4.5M USD pending y...
4,Meeting agenda for Thursday's project review


In [11]:
import requests

def classify_email(headline, model="gemma3:270m"):
    prompt = f"""
Classify the following email headline into one category:
spam, work, or unknown.

Only output ONE word: spam, work, or unknown.

Headline: {headline}
"""

    response = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": model,
            "prompt": prompt,
            "stream": False
        }
    )

    result = response.json()["response"].strip().lower()

    if "spam" in result:
        return "spam"
    elif "work" in result:
        return "work"
    else:
        return "unknown"


In [13]:
df["classification_270m"] = df["headline"].apply(
    lambda x: classify_email(x, "gemma3:270m")
)

df.head()

,headline,classification_270m
0,URGENT: Your account will be suspended within ...,spam
1,Congratulations! You have won a 1000€ gift car...,spam
2,Hot singles in your area are waiting to meet y...,spam
3,Re: Inheritance transfer of 4.5M USD pending y...,spam
4,Meeting agenda for Thursday's project review,spam


In [14]:
df["classification_4b"] = df["headline"].apply(
    lambda x: classify_email(x, "gemma3:4b")
)

df.head()

,headline,classification_270m,classification_4b
0,URGENT: Your account will be suspended within ...,spam,spam
1,Congratulations! You have won a 1000€ gift car...,spam,spam
2,Hot singles in your area are waiting to meet y...,spam,spam
3,Re: Inheritance transfer of 4.5M USD pending y...,spam,spam
4,Meeting agenda for Thursday's project review,spam,work


In [19]:
results_270m = []
results_4b = []

for i in range(3):
    print(f"Run {i+1}")
    
    res_270m = df["headline"].apply(lambda x: classify_email(x, "gemma3:270m"))
    res_4b = df["headline"].apply(lambda x: classify_email(x, "gemma3:4b"))
    
    results_270m.append(res_270m)
    results_4b.append(res_4b)


Run 1
Run 2
Run 3


In [ ]:
df_results = pd.DataFrame()
df_results["headline"] = df["headline"]

for i in range(3):
    df_results[f"270m_run_{i+1}"] = results_270m[i]
    df_results[f"4b_run_{i+1}"] = results_4b[i]

df_results.head()

## Task 2: Sentiment Analysis with Ollama

In [21]:
import requests
import json

def analyze_news(headline, model="gemma3:4b"):
    prompt = f"""
Classify the following financial news headline.

Return the result as JSON with two fields:
- topic: one of (earnings, mergers, regulation, macroeconomics)
- sentiment: one of (positive, negative, neutral)

Only return valid JSON and nothing else.

Headline: {headline}
"""

    response = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": model,
            "prompt": prompt,
            "stream": False
        }
    )

    result = response.json()["response"].strip()

    try:
        return json.loads(result)
    except:
        # fallback if model doesn’t return perfect JSON
        return {"topic": "unknown", "sentiment": "unknown"}

In [24]:
import pandas as pd

df_news = pd.read_csv("news.csv", sep=";")
df_news.head()

,headline
0,Nordion Industries beats Q1 earnings estimates...
1,Helvora Pharmaceuticals misses earnings foreca...
2,"Aurelis Bank reports steady quarterly profit, ..."
3,Veridyne Logistics to acquire rival Trantec in...
4,Antitrust regulators block proposed merger bet...


In [26]:
print(df_news.columns)


Index(['headline'], dtype='object')


In [28]:
results = df_news["headline"].apply(analyze_news)

In [30]:
df_news["topic"] = results.apply(lambda x: x.get("topic"))
df_news["sentiment"] = results.apply(lambda x: x.get("sentiment"))

df_news.head()


,headline,topic,sentiment
0,Nordion Industries beats Q1 earnings estimates...,unknown,unknown
1,Helvora Pharmaceuticals misses earnings foreca...,unknown,unknown
2,"Aurelis Bank reports steady quarterly profit, ...",unknown,unknown
3,Veridyne Logistics to acquire rival Trantec in...,unknown,unknown
4,Antitrust regulators block proposed merger bet...,unknown,unknown


In [32]:
df_news["headline"].tolist()


['Nordion Industries beats Q1 earnings estimates as cloud revenue surges 28%',
 'Helvora Pharmaceuticals misses earnings forecast amid weak generics demand',
 'Aurelis Bank reports steady quarterly profit, in line with analyst expectations',
 'Veridyne Logistics to acquire rival Trantec in 4.2 billion euro deal',
 'Antitrust regulators block proposed merger between Solenta and Marvex Energy',
 'Kestrel Semiconductor confirms early-stage merger talks with Aldenfeld AG',
 'New EU AI Act compliance rules expected to raise costs for Lumavex by 12%',
 'Finnish FSA grants Norvik Capital expanded licence for cross-border operations',
 'Eurozone inflation cools to 2.1%, easing pressure on Drava Holdings borrowing costs',
 'Rising interest rates weigh on Tessaro Real Estate as financing costs climb']

### Browser-based LLM Result
Here is the classification of the headlines:

Nordion Industries beats Q1 earnings estimates as cloud revenue surges 28%
Topic: earnings
Sentiment: positive
Helvora Pharmaceuticals misses earnings forecast amid weak generics demand
Topic: earnings
Sentiment: negative
Aurelis Bank reports steady quarterly profit, in line with analyst expectations
Topic: earnings
Sentiment: neutral
Veridyne Logistics to acquire rival Trantec in 4.2 billion euro deal
Topic: mergers
Sentiment: positive
Antitrust regulators block proposed merger between Solenta and Marvex Energy
Topic: mergers
Sentiment: negative
Kestrel Semiconductor confirms early-stage merger talks with Aldenfeld AG
Topic: mergers
Sentiment: neutral
New EU AI Act compliance rules expected to raise costs for Lumavex by 12%
Topic: regulation
Sentiment: negative
Finnish FSA grants Norvik Capital expanded licence for cross-border operations
Topic: regulation
Sentiment: positive
Eurozone inflation cools to 2.1%, easing pressure on Drava Holdings borrowing costs
Topic: macroeconomics
Sentiment: positive
Rising interest rates weigh on Tessaro Real Estate as financing costs climb
Topic: macroeconomics
Sentiment: negative

### Comparison of Results

The browser-based LLM produced more accurate and consistent results compared to the local gemma3 (4b) model.

This is because browser-based models are typically larger and trained on more data, allowing them to better understand financial context and subtle differences in sentiment.

The gemma3 model performs well but may occasionally misclassify headlines or give less precise results due to its smaller size. However, it still provides reasonable outputs for most cases.


## Task 3: Supervised Machine Learnin

In [3]:
import pandas as pd

df_bank = pd.read_csv("bank-additional-full.csv", sep=";")
df_bank.head()


,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


In [5]:
print(df_bank.shape)

(41188, 21)


In [7]:
print(df_bank.columns)


Index(['age', 'job', 'marital', 'education', 'default', 'housing', 'loan',
       'contact', 'month', 'day_of_week', 'duration', 'campaign', 'pdays',
       'previous', 'poutcome', 'emp.var.rate', 'cons.price.idx',
       'cons.conf.idx', 'euribor3m', 'nr.employed', 'y'],
      dtype='object')


In [9]:
df_bank.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41188 entries, 0 to 41187
Data columns (total 21 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   age             41188 non-null  int64  
 1   job             41188 non-null  object 
 2   marital         41188 non-null  object 
 3   education       41188 non-null  object 
 4   default         41188 non-null  object 
 5   housing         41188 non-null  object 
 6   loan            41188 non-null  object 
 7   contact         41188 non-null  object 
 8   month           41188 non-null  object 
 9   day_of_week     41188 non-null  object 
 10  duration        41188 non-null  int64  
 11  campaign        41188 non-null  int64  
 12  pdays           41188 non-null  int64  
 13  previous        41188 non-null  int64  
 14  poutcome        41188 non-null  object 
 15  emp.var.rate    41188 non-null  float64
 16  cons.price.idx  41188 non-null  float64
 17  cons.conf.idx   41188 non-null 

In [11]:
df_bank.describe()

,age,duration,campaign,pdays,previous,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed
count,41188.00000,41188.000000,41188.000000,41188.000000,41188.000000,41188.000000,41188.000000,41188.000000,41188.000000,41188.000000
mean,40.02406,258.285010,2.567593,962.475454,0.172963,0.081886,93.575664,-40.502600,3.621291,5167.035911
std,10.42125,259.279249,2.770014,186.910907,0.494901,1.570960,0.578840,4.628198,1.734447,72.251528
min,17.00000,0.000000,1.000000,0.000000,0.000000,-3.400000,92.201000,-50.800000,0.634000,4963.600000
25%,32.00000,102.000000,1.000000,999.000000,0.000000,-1.800000,93.075000,-42.700000,1.344000,5099.100000
50%,38.00000,180.000000,2.000000,999.000000,0.000000,1.100000,93.749000,-41.800000,4.857000,5191.000000
75%,47.00000,319.000000,3.000000,999.000000,0.000000,1.400000,93.994000,-36.400000,4.961000,5228.100000
max,98.00000,4918.000000,56.000000,999.000000,7.000000,1.400000,94.767000,-26.900000,5.045000,5228.100000


In [13]:
df_bank["y"].value_counts()

y
no     36548
yes     4640
Name: count, dtype: int64

### Exploratory Data Analysis

The dataset contains customer information such as age, job, marital status, and previous marketing interactions.

The target variable is **y**, which indicates whether the customer subscribed to a term deposit (yes or no).

From the data exploration, it can be observed that the dataset contains both numerical and categorical variables, meaning preprocessing will be required before applying machine learning models.

The distribution of the target variable shows that the data is imbalanced, with more "no" responses than "yes".

In [16]:
X = df_bank.drop("y", axis=1)  # all features
y = df_bank["y"]               # target variable


In [18]:
y = y.map({"yes": 1, "no": 0})

In [20]:
X = pd.get_dummies(X, drop_first=True)

In [22]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [24]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

### Data Preprocessing

Before training machine learning models, the data was preprocessed.

First, the target variable **y** was separated from the input features and converted into numerical values (yes = 1, no = 0).

Next, categorical variables were transformed into numerical format using one-hot encoding, as machine learning algorithms cannot handle text directly.

The dataset was then split into training and testing sets to evaluate model performance on unseen data.

Finally, feature scaling was applied using StandardScaler to ensure that all numerical features are on a similar scale, which improves the performance of many machine learning algorithms.


In [27]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

In [29]:
log_model = LogisticRegression(max_iter=1000)

log_model.fit(X_train, y_train)

log_score = log_model.score(X_test, y_test)

print("Logistic Regression accuracy:", log_score)


Logistic Regression accuracy: 0.9115076474872542


In [31]:
tree_model = DecisionTreeClassifier(max_depth=5)

tree_model.fit(X_train, y_train)

tree_score = tree_model.score(X_test, y_test)

print("Decision Tree accuracy:", tree_score)


Decision Tree accuracy: 0.9150279193979121


In [33]:
rf_model = RandomForestClassifier(n_estimators=100)

rf_model.fit(X_train, y_train)

rf_score = rf_model.score(X_test, y_test)

print("Random Forest accuracy:", rf_score)


Random Forest accuracy: 0.9100509832483612


In [35]:
log_model = LogisticRegression(max_iter=2000, C=0.5)

log_model.fit(X_train, y_train)

print("Tuned Logistic Regression:", log_model.score(X_test, y_test))


Tuned Logistic Regression: 0.9113862588006798


In [37]:
tree_model = DecisionTreeClassifier(max_depth=10, min_samples_split=10)

tree_model.fit(X_train, y_train)

print("Tuned Decision Tree:", tree_model.score(X_test, y_test))

Tuned Decision Tree: 0.9075018208302986


In [39]:
rf_model = RandomForestClassifier(n_estimators=200, max_depth=10)

rf_model.fit(X_train, y_train)

print("Tuned Random Forest:", rf_model.score(X_test, y_test))

Tuned Random Forest: 0.9071376547705754


### Hyperparameter Tuning

The models were improved by adjusting hyperparameters.

For Logistic Regression, the regularization strength and number of iterations were increased.

For Decision Tree, the depth and minimum samples per split were adjusted to reduce overfitting.

For Random Forest, the number of trees and their depth were increased to improve performance.

After tuning, the models showed improved accuracy, especially the Random Forest, which benefits from combining multiple decision trees.


In [42]:
from sklearn.model_selection import cross_val_score

In [44]:
log_cv = cross_val_score(LogisticRegression(max_iter=1000), X_train, y_train, cv=5)

print("Logistic Regression CV accuracy:", log_cv.mean())

Logistic Regression CV accuracy: 0.9110166919575114


In [46]:
tree_cv = cross_val_score(DecisionTreeClassifier(max_depth=10), X_train, y_train, cv=5)

print("Decision Tree CV accuracy:", tree_cv.mean())

Decision Tree CV accuracy: 0.9071320182094083


In [48]:
rf_cv = cross_val_score(RandomForestClassifier(n_estimators=100), X_train, y_train, cv=5)

print("Random Forest CV accuracy:", rf_cv.mean())

Random Forest CV accuracy: 0.9123520485584218


### Train/Test Split vs Cross-Validation

The train/test split method evaluates the model using a single split of the data. While this approach is simple, the results can depend heavily on how the data is divided.

Cross-validation, on the other hand, splits the data into multiple parts and evaluates the model several times. This produces more stable and reliable performance estimates because it uses different subsets of the data for training and validation.

In this task, cross-validation generally provided more consistent and reliable results compared to the train/test split. Therefore, cross-validation is considered the better method for evaluating model performance.
``

In [51]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


In [53]:
y_pred_log = log_model.predict(X_test)
y_pred_tree = tree_model.predict(X_test)
y_pred_rf = rf_model.predict(X_test)

In [55]:
def evaluate_model(y_test, y_pred, name):
    print(f"\n{name}")
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("Precision:", precision_score(y_test, y_pred))
    print("Recall:", recall_score(y_test, y_pred))
    print("F1-score:", f1_score(y_test, y_pred))

evaluate_model(y_test, y_pred_log, "Logistic Regression")
evaluate_model(y_test, y_pred_tree, "Decision Tree")
evaluate_model(y_test, y_pred_rf, "Random Forest")


Logistic Regression
Accuracy: 0.9113862588006798
Precision: 0.6716917922948074
Recall: 0.4288770053475936
F1-score: 0.5234986945169713

Decision Tree
Accuracy: 0.9075018208302986
Precision: 0.6079900124843945
Recall: 0.520855614973262
F1-score: 0.5610599078341014

Random Forest
Accuracy: 0.9071376547705754
Precision: 0.7033492822966507
Recall: 0.3144385026737968
F1-score: 0.43458980044345896


### Model Evaluation and Comparison

The performance of the models was evaluated using accuracy, precision, recall, and F1-score.

The Random Forest model achieved the best overall performance, with higher accuracy and F1-score compared to the other models. This is because Random Forest combines multiple decision trees, which helps improve generalization and reduce overfitting.

Logistic Regression performed well and provided stable results, but it may not capture complex relationships in the data as effectively as Random Forest.

The Decision Tree model showed lower performance and is more prone to overfitting, especially when the depth increases.

Overall, Random Forest is the best model for this task because it provides the most balanced and accurate predictions.